In [1]:
import os
from dotenv import load_dotenv
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import ChatPromptTemplate

import sqlite3
import pandas as pd

In [2]:
# Load environment variables from .env file
load_dotenv()


True

In [3]:
# Load data from the database
def load_reports_table(db_path: str = "fakesharks.db") -> pd.DataFrame:
    """
    Loads the 'reports' table from the specified SQLite database into a pandas DataFrame.

    Args:
        db_path (str): Path to the SQLite database file. Defaults to 'fakesharks.db'.

    Returns:
        pd.DataFrame: DataFrame containing all rows from the 'reports' table.

    Raises:
        sqlite3.Error: If there is an issue connecting to or querying the database.
    """
    try:
        # Establish a connection to the SQLite database
        conn = sqlite3.connect(db_path)
        # Read the 'reports' table into a DataFrame
        df = pd.read_sql_query("SELECT * FROM reports", conn)
    except sqlite3.Error as e:
        # Log or handle database errors as needed
        print(f"Database error: {e}")
        df = pd.DataFrame()  # Return empty DataFrame on error
    finally:
        # Ensure the connection is closed even if an error occurs
        if 'conn' in locals():
            conn.close()
    return df

# load the data
df = load_reports_table()
print(f"Reports data loaded.. shape: {df.shape}")

Reports data loaded.. shape: (9, 10)


In [4]:
# Let's look at the data
df.head()

,id,timestamp,lat,lon,shark_type,body_part,severity,description,long_description,survived
0,1,2026-01-27 04:10:56.721094,47.8000,-122.4200,Tiger Shark,Torso,1,Oy! Bit by a shark on my paddleboard!,None,1
1,2,2026-01-27 04:12:29.666591,19.7900,-155.0300,Great White,Arm,4,I hope they know how to paddle that surfboard ...,None,1
2,3,2026-01-27 04:38:41.917103,20.6100,-105.7500,Mako,Head,3,Mako shark tried to Make-o out. Did not go well.,None,1
3,4,2026-02-08 04:42:33.429758,-40.2795,173.9795,Great White,Multiple,1,"A tour de kiwi. This great white tried arms, l...",None,1
4,5,2026-02-08 04:46:42.415513,24.5171,-76.9043,Tiger Shark,Arm,5,Clever girl... shallow water!,None,1


In [16]:
# here I'm testing out the ability to geocode a lat/long and turn it into some sort of a more description to pass to the LLM.
# Note the usage policy, we need to wait 1 second between calls to the API
# Policy URL: https://operations.osmfoundation.org/policies/nominatim/

import time
from geopy.geocoders import Nominatim

# Let's try them all
for _ in range(df.shape[0]):

    # Get the lat/long
    lat = df.loc[_, "lat"]
    lon = df.loc[_, "lon"]

    print(f"Now Geocoding lat: {lat}, lon: {lon}")

    try:
        geolocator = Nominatim(user_agent="fakesharks_app")
        location = geolocator.reverse((lat, lon), language="en")
        if location.address is None:
            print(f"No address found for lat: {lat}, lon: {lon}")
            print(f"location.raw: {location.raw}")
        else:
            print(f"location.address: {location.address}")
    except Exception as e:
        print(f"Error geocoding lat: {lat}, lon: {lon} - {e}")

    time.sleep(1.25) # wait a bit more than 1 second


# Keep this code, but ultimately not worth pursing if it doesn't geocode oceans well. 


Now Geocoding lat: 47.8, lon: -122.42
location.address: Snohomish County, Washington, United States
Now Geocoding lat: 19.79, lon: -155.03
location.address: Hawaiʻi County, Hawaii, United States
Now Geocoding lat: 20.61, lon: -105.75
location.address: Mexico
Now Geocoding lat: -40.2795, lon: 173.9795
Error geocoding lat: -40.2795, lon: 173.9795 - 'NoneType' object has no attribute 'address'
Now Geocoding lat: 24.5171, lon: -76.9043
location.address: Exuma, Bahamas
Now Geocoding lat: -42.8115, lon: -103.0078
Error geocoding lat: -42.8115, lon: -103.0078 - 'NoneType' object has no attribute 'address'
Now Geocoding lat: -39.9097, lon: 168.3984
Error geocoding lat: -39.9097, lon: 168.3984 - 'NoneType' object has no attribute 'address'
Now Geocoding lat: 74.4964, lon: -60.6445
Error geocoding lat: 74.4964, lon: -60.6445 - 'NoneType' object has no attribute 'address'
Now Geocoding lat: 26.4312, lon: -37.6172
Error geocoding lat: 26.4312, lon: -37.6172 - 'NoneType' object has no attribute 'ad

In [21]:

SYSTEM_PROMPT = """\
You are an up-and-coming yet sarcastic journalist. You have been assigned
to write an article about a shark attack. The article should take the details
given and expand them into a longer article. The article should be
brief and about 90 words long. Do not cut off your article for brevity, ensure it is complete.
Your goal is to make cheeky jokes about the situation. All situations will be shark attacks.
Use the shark species, the injured body part, the severity of the attack, and whether the
victim survived to add colour and dark humour to your writing. Do not invent details beyond
what is provided.
"""

human_prompt = """\
Short description: {description}

Shark species: {shark_type}
Body part injured: {body_part}
Severity: {severity}
Victim survived: {survived}

Article:"""

# Create the prompt template
prompt = ChatPromptTemplate.from_messages([("system", SYSTEM_PROMPT), ("human", human_prompt)])

# Set up the endpoint
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    temperature=0.7,
    max_new_tokens=250,
    huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN"),
)

chat_model = ChatHuggingFace(llm=llm)

chain = prompt | chat_model


In [14]:
# Quick test — invoke the chat model directly
response = chat_model.invoke("Explain what LangChain is in one short paragraph.")
print(response.content)


LangChain is a framework designed to enable the integration of language models with various applications and systems, facilitating the creation of more sophisticated and context-aware AI tools. It provides a suite of libraries and utilities that allow developers to easily chain together different components, such as language models, data sources, and user interfaces, to build powerful AI applications.


In [22]:
# Test with the first row from the loaded data
row = df.iloc[3]
response = chain.invoke({
    "description": row["description"],
    "shark_type": row["shark_type"],
    "body_part": row["body_part"],
    "severity": row["severity"],
    "survived": row["survived"],
})
print(response.content)

print("Word count:", len(response.content.split()))  # Check the word count of the generated article

In a tale that would make any travel brochure cringe, a Kiwi tourist found themselves the unwitting subject of a shark attack by a Great White. The "tour de kiwi," as it's being dubbed, saw the shark meticulously attempting to sample the local cuisine by attacking the arms, legs, and torso. Despite the shark's best efforts, our Kiwi hero managed to survive, much to the relief of his sunscreen-less friends. Who knew the Great White was such a culinary critic? Maybe next time, the shark should stick to its usual fishy fare.
Word count: 92


In [24]:
# Let's get descriptions for all rows in reports
def generate_long_descriptions(df: pd.DataFrame) -> pd.DataFrame:
    """
    Generates long descriptions for each row in the given DataFrame using a language model.

    Args:
        df (pd.DataFrame): DataFrame containing the reports data.

    Returns:
        pd.DataFrame: DataFrame with an additional column 'long_description' containing the generated articles.
    """
    try:
        # Create a copy of the DataFrame to avoid modifying the original
        df_copy = df.copy()
        # Initialize an empty list to store the generated long descriptions
        long_descriptions = []
        
        # Iterate over each row in the DataFrame
        for _, row in df_copy.iterrows():
            # Prepare the input for the language model
            input_data = {
                "description": row["description"],
                "shark_type": row["shark_type"],
                "body_part": row["body_part"],
                "severity": row["severity"],
                "survived": row["survived"],
            }
            # Generate the long description using the chain
            response = chain.invoke(input_data)
            long_descriptions.append(response.content)
        
        # Add the generated long descriptions as a new column in the DataFrame
        df_copy["long_description"] = long_descriptions
    except Exception as e:
        print(f"Error generating long descriptions: {e}")
        df_copy["long_description"] = None  # Set to None on error
    return df_copy

df = generate_long_descriptions(df)
print(df[["description", "long_description"]].head())

                                         description  \
0              Oy! Bit by a shark on my paddleboard!   
1  I hope they know how to paddle that surfboard ...   
2   Mako shark tried to Make-o out. Did not go well.   
3  A tour de kiwi. This great white tried arms, l...   
4                      Clever girl... shallow water!   

                                    long_description  
0  Oy! Bit by a tiger shark on my paddleboard! If...  
1  In a shocking turn of events, a surfer found t...  
2  In a shocking turn of events that left the vic...  
3  In a bizarre turn of events straight out of a ...  
4  In a stroke of pure luck—or maybe pure stupidi...  


In [29]:
def save_to_db(df: pd.DataFrame, table_name: str = "reports_w_test_articles", db_path: str = "fakesharks.db") -> None:
    """
    Saves the given DataFrame to the specified SQLite database table.
    Replaces the table if it already exists.

    Args:
        df (pd.DataFrame): DataFrame to save.
        table_name (str): Name of the destination table.
        db_path (str): Path to the SQLite database file.
    """
    try:
        conn = sqlite3.connect(db_path)
        df.to_sql(table_name, conn, if_exists="replace", index=False)
        print(f"Saved {len(df)} rows to table '{table_name}' in {db_path}")
    except sqlite3.Error as e:
        print(f"Database error: {e}")
    finally:
        if 'conn' in locals():
            conn.close()

save_to_db(df)


Saved 9 rows to table 'reports_w_test_articles' in fakesharks.db
